# L04 · MDP와 Bellman equation

## Goal

- state·action·transition·reward를 정의한다
- Bellman backup을 계산한다
- value iteration 결과를 해석한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L04:toy:42").hexdigest()
print(f"lesson=L04 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L04 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:42f1ff9de0bc274e282f19496e2b5a797a2a3c9f9d59d57b348abed6cb1058d0 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: bandit → **MDP·Bellman** → MC/TD/Q-learning

$$V_{k+1}(s)=\max_a\sum_{s'}P(s'\mid s,a)\left[r+\gamma V_k(s')\right]$$

MDP는 미래가 현재 state와 action으로 충분하다는 모델입니다. Bellman backup은 장기 return 문제를 한 step reward와 다음 state value로 재귀 분해합니다. terminal state에는 미래가 없으므로 bootstrap value가 0입니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** goal 바로 옆 state의 value는 terminal value 0보다 클까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>goal로 들어갈 때 reward를 받으므로 큽니다. terminal 자체에서 다시 reward를 받는 것은 아닙니다.</details>

In [2]:
from rl_study.algorithms.tabular import gridworld_model, value_iteration
from rl_study.envs import TinyGridWorld
grid = TinyGridWorld()
transitions, rewards, terminal = gridworld_model(grid)
dp_result = value_iteration(transitions, rewards, terminal, gamma=0.99)
print("values")
print(dp_result.values.reshape(4, 4).round(decimals=3))
print({"iterations": dp_result.iterations,
       "converged": dp_result.converged,
       "terminal_value": float(dp_result.values[-1])})

values
tensor([[0.9020, 0.9210, 0.9410, 0.9600],
        [0.9210, 0.9410, 0.9600, 0.9800],
        [0.9410, 0.9600, 0.9800, 1.0000],
        [0.9600, 0.9800, 1.0000, 0.0000]])
{'iterations': 7, 'converged': True, 'terminal_value': 0.0}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** transition model이 알려진 작은 grid에서는 value iteration이 정확한 기준선을 제공합니다. model-free 학습과 비교할 때 환경·gamma 오류를 먼저 분리할 수 있습니다.

**흔한 함정:** terminal에서 bootstrap하면 goal에 머물며 reward를 무한 반복하는 잘못된 MDP가 됩니다. terminal value 0 검사가 경계를 고정합니다. 회귀 test: `test_value_iteration_terminal_value_is_zero`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert dp_result.converged and dp_result.values[-1].item() == 0.0
print("checks=passed")

checks=passed


**회상 문제:** Bellman expectation equation과 optimality equation에서 `max`의 유무는 무엇을 뜻하나요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** value iteration은 7회에 수렴했고 terminal value는 정확히 0입니다. goal에 가까운 비종료 state일수록 할인된 value가 커집니다.
- 실제 확인: `test_value_iteration_terminal_value_is_zero`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L05에서는 transition model 없이 실제 trajectory로 이 Bellman target을 추정합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`